<a href="https://colab.research.google.com/github/Exper626/irpWhiteTea/blob/main/Interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics gradio scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00


In [3]:
import gradio as gr
import numpy as np
import pickle
import cv2
import torch
from ultralytics import YOLO
from PIL import Image

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Load models

In [4]:
BEST_MODEL    = '/content/drive/MyDrive/yolo_weights/yolov8n_whitetea_best.pt'
ANN_PATH      = '/content/drive/MyDrive/ann_pipeline/ann_model.pkl'
SCALER_PATH   = '/content/drive/MyDrive/ann_pipeline/scaler.pkl'

yolo_model = YOLO(BEST_MODEL)

with open(ANN_PATH,    'rb') as f: ann    = pickle.load(f)
with open(SCALER_PATH, 'rb') as f: scaler = pickle.load(f)

print('Models loaded successfully.')

Models loaded successfully.


 Inference function

In [5]:
def predict_harvest_readiness(image):
    """
    Full pipeline:
    1. Run YOLO detection on uploaded image
    2. Extract detection-derived features
    3. Run ANN classification
    4. Return annotated image + readiness report
    """

    # Convert PIL → numpy BGR for YOLO
    img_np  = np.array(image)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    # ── YOLO inference ───────────────────────────────────────────────────────
    results = yolo_model.predict(
        source=img_bgr,
        conf=0.25,
        verbose=False
    )[0]

    boxes = results.boxes
    h, w  = img_bgr.shape[:2]

    # Draw detections
    annotated = img_bgr.copy()
    bud_count = 0

    if boxes is not None and len(boxes) > 0:
        bud_count = len(boxes)
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 200, 50), 2)
            cv2.putText(
                annotated,
                f'bud {conf:.2f}',
                (x1, max(y1-8, 0)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (0, 200, 50), 1
            )

    # ── Feature extraction ───────────────────────────────────────────────────
    if boxes is not None and len(boxes) > 0:
        xywhn = boxes.xywhn.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        cx    = xywhn[:, 0]
        cy    = xywhn[:, 1]
        bw    = xywhn[:, 2]
        bh    = xywhn[:, 3]
        areas = bw * bh
        dist_from_centre = np.sqrt((cx - 0.5)**2 + (cy - 0.5)**2)

        features = np.array([[
            bud_count,
            float(np.mean(confs)),
            float(np.mean(areas)),
            float(bud_count),
            float(np.std(cx)) if bud_count > 1 else 0.0,
            float(np.mean(dist_from_centre))
        ]])
    else:
        features = np.zeros((1, 6))

    # ── ANN prediction ───────────────────────────────────────────────────────
    features_scaled = scaler.transform(features)
    prediction      = ann.predict(features_scaled)[0]
    probabilities   = ann.predict_proba(features_scaled)[0]

    label = 'Approaching Ready' if prediction == 1 else 'Not Ready'
    conf_pct = probabilities[prediction] * 100

    # ── Build readiness report ───────────────────────────────────────────────
    report = f"""
🍃 WHITE TEA BUD DETECTION REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📊 Detection Results
  Buds detected:        {bud_count}
  Model:                YOLOv8n (best mAP50)
  Confidence threshold: 0.25

🧠 Harvest Readiness Assessment
  Status:               {label}
  Confidence:           {conf_pct:.1f}%

⚠️  IMPORTANT NOTE
  This assessment is a feasibility demonstration.
  Ground-truth harvest readiness labels were not
  available for full ANN validation within this
  project scope. Expert field assessment should
  be used for operational harvest decisions.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CM4605 Final Year Project — Hanaa Azhar
  Robert Gordon University / IIT
    """

    # Convert annotated image back to RGB for Gradio
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    return Image.fromarray(annotated_rgb), report

Build and launch Gradio interface

In [6]:
with gr.Blocks(title='White Tea Bud Detection', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🍃 White Tea Bud Detection & Harvest Readiness Assessment
    **CM4605 Individual Research Project — Hanaa Azhar (2237922)**
    *Robert Gordon University in collaboration with IIT Sri Lanka*

    ---
    Upload a field image of a white tea plant. The system will:
    1. Detect individual white tea buds using YOLOv8n
    2. Extract detection features (count, density, spatial distribution)
    3. Classify harvest readiness using an ANN pipeline

    > ⚠️ This is a research feasibility demonstration. Operational harvest
    > decisions should always involve expert field assessment.
    """)

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(
                type='pil',
                label='Upload Field Image'
            )
            submit_btn = gr.Button(
                '🔍 Detect & Assess',
                variant='primary'
            )

        with gr.Column():
            output_image = gr.Image(
                label='Detection Results'
            )
            output_text = gr.Textbox(
                label='Harvest Readiness Report',
                lines=20
            )

    submit_btn.click(
        fn=predict_harvest_readiness,
        inputs=input_image,
        outputs=[output_image, output_text]
    )

    gr.Markdown("""
    ---
    **Model:** YOLOv8n trained on 72 annotated white tea bud images (Hadungoda Estate, Sri Lanka)
    **Dataset validated by:** Asia Siyaka Commodities PLC, Colombo
    """)

demo.launch(share=True, debug=False)

/tmp/ipykernel_9301/3515112456.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='White Tea Bud Detection', theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb97c9ae4949a15f2b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
